In [ ]:
import pickle

from Bio import SeqIO
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, auc, f1_score, precision_score, recall_score, roc_curve

# Train

In [ ]:
%%bash
export MSA_data_folder='../data/processed/'
export MSA_list='../data/processed/sars_nsp5_site_8aa_mapping.csv'
export MSA_weights_location='../checkpoints/sars_nsp5_8aa/weights'
export VAE_checkpoint_location='../checkpoints/sars_nsp5_8aa'
export model_name_suffix='sars_nsp5_8aa'
export model_parameters_location='../src/EVE/EVE/default_model_params.json'
export training_logs_location='../logs/sars_nsp5_8aa'
export protein_index=0

mkdir -p ${MSA_weights_location}
mkdir -p ${training_logs_location}

python ../src/EVE/train_VAE.py \
    --MSA_data_folder ${MSA_data_folder} \
    --MSA_list ${MSA_list} \
    --protein_index ${protein_index} \
    --MSA_weights_location ${MSA_weights_location} \
    --VAE_checkpoint_location ${VAE_checkpoint_location} \
    --model_name_suffix ${model_name_suffix} \
    --model_parameters_location ${model_parameters_location} \
    --training_logs_location ${training_logs_location}

# Test

In [ ]:
%%bash
export MSA_data_folder='../data/processed/'
export MSA_list='../data/processed/nsp5_testset_8aa_mapping.csv'
export MSA_weights_location='../checkpoints/sars_nsp5_8aa/weights/'
export VAE_checkpoint_location='../checkpoints/sars_nsp5_8aa'
export model_name_suffix='sars_nsp5_8aa'
export model_parameters_location='../src/EVE/EVE/default_model_params.json'
export training_logs_location='../logs/sars_nsp5_8aa/'
export protein_index=0

export computation_mode='all_singles'
export all_singles_mutations_folder='../results/EVE/sars_nsp_5_8aa_testset/mutations'
export output_evol_indices_location='../results/EVE/sars_nsp_5_8aa_testset/evol_indices'
export num_samples_compute_evol_indices=20000
export batch_size=2048

mkdir -p ${all_singles_mutations_folder}
mkdir -p ${output_evol_indices_location}

python ../src/EVE/compute_evol_indices.py \
    --MSA_data_folder ${MSA_data_folder} \
    --MSA_list ${MSA_list} \
    --protein_index ${protein_index} \
    --MSA_weights_location ${MSA_weights_location} \
    --VAE_checkpoint_location ${VAE_checkpoint_location} \
    --model_name_suffix ${model_name_suffix} \
    --model_parameters_location ${model_parameters_location} \
    --computation_mode ${computation_mode} \
    --all_singles_mutations_folder ${all_singles_mutations_folder} \
    --output_evol_indices_location ${output_evol_indices_location} \
    --num_samples_compute_evol_indices ${num_samples_compute_evol_indices} \
    --batch_size ${batch_size}

## compute evol indices

In [ ]:
# convert fasta to df
headers = []
sequences = []

for record in SeqIO.parse(f'../data/processed/nsp5_testset_8aa.fasta', 'fasta'):
    headers.append(record.id)
    sequences.append(str(record.seq))

test_set_dict = {
    'header': headers,
    'sequence': sequences
}

test_set_df = pd.DataFrame.from_dict(test_set_dict)
test_set_df

In [ ]:
def calculate_sequence_score(csv_path, wt_sequence, new_sequence):
    evol_data = pd.read_csv(csv_path)

    if wt_sequence == new_sequence:
        wt_score = evol_data[evol_data['mutations'] == 'wt']['evol_indices'].values[0]
        return wt_score

    # find out all mutations
    mutations = []
    for i in range(len(wt_sequence)):
        if wt_sequence[i] != new_sequence[i]:
            # format as T1A (original amino acid + position + new amino acid)
            mutation = f'{wt_sequence[i]}{i+1}{new_sequence[i]}'
            mutations.append(mutation)

    # compute score (all mutations' scores' sum)
    total_score = 0.0
    for mutation in mutations:
        # find the corresponding score in the CSV
        mutation_row = evol_data[evol_data['mutations'] == mutation]
        total_score += mutation_row['evol_indices'].values[0]
    return total_score

In [ ]:
# WT using in training
wt_seq = next(SeqIO.parse(f'../data/processed/nsp5_testset_8aa.fasta', 'fasta')).seq
wt_seq

In [ ]:
test_set_df['score'] = test_set_df['sequence'].apply(
    lambda x: calculate_sequence_score('../results/EVE/sars_nsp_5_8aa_testset/evol_indices/nsp5_20000_samples.csv',
                                       str(wt_seq),
                                       x)
)
test_set_df

## ROC curve

In [ ]:
test_set_df['label'] = test_set_df['header'].apply(lambda x: 1 if 'pos' in x else 0)
test_set_df['neg_score'] = -test_set_df['score']
test_set_df

In [ ]:
fpr, tpr, _ = roc_curve(test_set_df['label'], test_set_df['neg_score'])
roc_auc = auc(fpr, tpr)
plt.figure()

plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()

exclude two weak positives

In [ ]:
sub_test_set_df = test_set_df.loc[~test_set_df['header'].str.contains('weak')]

fpr, tpr, _ = roc_curve(sub_test_set_df['label'], sub_test_set_df['neg_score'])
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()

## Classification model

In [ ]:
model = LogisticRegression(random_state=138)
model.fit(test_set_df[['score']], test_set_df['label'])
y_pred = model.predict(test_set_df[['score']])
accuracy = accuracy_score(test_set_df['label'], y_pred)
f1 = f1_score(test_set_df['label'], y_pred)
precision = precision_score(test_set_df['label'], y_pred, zero_division=0)
recall = recall_score(test_set_df['label'], y_pred, zero_division=0)
accuracy, f1, precision, recall

In [ ]:
fig = plt.figure()
ax = plt.gca()

plt.scatter(test_set_df['score'], test_set_df['label'])

# add logistic regression curve
x_range = np.linspace(min(test_set_df['score']), max(test_set_df['score']), 1000)
x_range_df = pd.DataFrame(x_range, columns=['score'])
y_proba_curve = model.predict_proba(x_range_df)[:, 1]
plt.plot(x_range, y_proba_curve)

# add decision_boundary
idx_nearest_to_half = np.abs(y_proba_curve - 0.5).argmin()
decision_boundary = x_range[idx_nearest_to_half]
plt.axvline(x=decision_boundary, color='#777777', linestyle='--', alpha=0.7)
plt.axhline(y=0.5, color='#777777', linestyle='--', alpha=0.7)

plt.xlabel('Evolutionary score')
plt.ylabel('Probability of activity')
plt.ylim(-0.05, 1.05)
x_min, x_max = min(test_set_df['score']), max(test_set_df['score'])
x_margin = (x_max - x_min) * 0.05
plt.xlim(x_min - x_margin, x_max + x_margin)

plt.tight_layout()
plt.show()

In [ ]:
decision_boundary

In [ ]:
with open('../checkpoints/logistic_model.pkl', 'wb') as file:
    pickle.dump(model, file)